In [4]:
import sys
import numpy as np
import scipy
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Critical compatibility check
from scipy.linalg import cholesky

print("\nNumPy and SciPy imports successful!")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.1.3
SciPy: 1.16.3
PyTorch: 2.11.0+cu130
CUDA Available: True
GPU: Tesla T4

NumPy and SciPy imports successful!


In [6]:
%pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    evaluate \
    huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [7]:

# ============================================================
# PHASE 6: QLoRA FINE-TUNING
# Centralized Imports
# ============================================================

# -------------------------
# Standard Python Libraries
# -------------------------
import os
import gc
import json
import time
import random
import warnings

from pathlib import Path

# -------------------------
# Data Processing
# -------------------------
import numpy as np
import pandas as pd

# -------------------------
# Visualization
# -------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# PyTorch
# -------------------------
import torch
import torch.nn as nn

# -------------------------
# Hugging Face Datasets
# -------------------------
from datasets import Dataset

# -------------------------
# Transformers
# -------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)

# -------------------------
# PEFT / LoRA
# -------------------------
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

# -------------------------
# TRL / Supervised Fine-Tuning
# -------------------------
from trl import SFTTrainer, SFTConfig

# -------------------------
# Evaluation
# -------------------------
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# Progress Monitoring
# -------------------------
from tqdm.auto import tqdm

print("All Phase 6 imports completed successfully!")


All Phase 6 imports completed successfully!


In [8]:

import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("========== ENVIRONMENT ==========")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

print("\n========== GPU ==========")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )


========== ENVIRONMENT ==========
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
Accelerate: 1.15.0
PEFT: 0.21.2
BitsAndBytes: 0.50.2
TRL: 1.18.1

========== GPU ==========
CUDA Available: True
GPU: Tesla T4
Total GPU Memory: 14.56 GB


In [10]:
# Load the training dataset
DIR_PATH = Path("/kaggle/input/datasets/dgowda11/sms-spam-cleaned")

TRAIN_PATH = DIR_PATH / "train.csv"
TEST_PATH = DIR_PATH / "test.csv"

assert TRAIN_PATH.exists(), "train path not found !"
assert TEST_PATH.exists(), "test path not found"

train_df  = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Tranining Shape :",train_df.shape)
print("Testing Shape :",test_df.shape)

display(train_df.head())

Tranining Shape : (4135, 2)
Testing Shape : (1034, 2)


,label,message
0,ham,Cool. So how come you havent been wined and di...
1,ham,Huh so late... Fr dinner?
2,ham,my ex-wife was not able to have kids. Do you w...
3,ham,Hope ur head doesn't hurt 2 much ! Am ploughin...
4,ham,Ryder unsold.now gibbs.


Supervised Fine-Tuning (SFT) Data Preparation.

In [11]:
#Loging to Hugging Face 
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve Hugging Face token from Kaggle Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Authenticate with Hugging Face
login(token=HF_TOKEN)

print("Hugging Face authentication successful!")

Hugging Face authentication successful!


In [12]:
# Load the Tokenizer 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)
print("Tokenizer loaded successfully!")
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Tokenizer: TokenizersBackend
Vocabulary size: 128000
EOS Token: <|eot_id|>


In [13]:
SYSTEM_PROMPT = """
You are an SMS classification assistant.

Your task is to classify SMS messages into one of two categories:
- HAM: A legitimate, normal message.
- SPAM: An unsolicited, fraudulent, promotional, or suspicious message.

Analyze the message carefully.

Respond with only one label:
HAM or SPAM.

Do not provide explanations or additional text.
"""

print(SYSTEM_PROMPT)


You are an SMS classification assistant.

Your task is to classify SMS messages into one of two categories:
- HAM: A legitimate, normal message.
- SPAM: An unsolicited, fraudulent, promotional, or suspicious message.

Analyze the message carefully.

Respond with only one label:
HAM or SPAM.

Do not provide explanations or additional text.



In [14]:
#Formatting Function
CHAT_DATE = "19 Nov 2026"
def format_sms(message,label):

    messages = [
        {
            "role":"system",
            "content": SYSTEM_PROMPT
        },
        {
            "role":"user",
            "content": f"message:{message}"
        },
        {
            "role":"assistant",
            "content": label.upper()
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt = False,
        date_string = CHAT_DATE
    )

In [15]:
sample_message = train_df.iloc[0]["message"]
sample_label = train_df.iloc[0]["label"]

formatted_example = format_sms(
    sample_message,
    sample_label
)

print(formatted_example)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 19 Nov 2026

You are an SMS classification assistant.

Your task is to classify SMS messages into one of two categories:
- HAM: A legitimate, normal message.
- SPAM: An unsolicited, fraudulent, promotional, or suspicious message.

Analyze the message carefully.

Respond with only one label:
HAM or SPAM.

Do not provide explanations or additional text.<|eot_id|><|start_header_id|>user<|end_header_id|>

message:Cool. So how come you havent been wined and dined before?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

HAM<|eot_id|>


Tokenization

In [18]:
def tokenize_sms_example(message, label):

    prompt_messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"message:{message}"
        }
    ]

    full_messages = prompt_messages + [
        {
            "role": "assistant",
            "content": label.upper()
        }
    ]

    # Tokenize prompt
    prompt_output = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        date_string=CHAT_DATE
    )

    # Tokenize complete conversation
    full_output = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        date_string=CHAT_DATE
    )

    # Normalize outputs into Python lists
    prompt_ids = (
        list(prompt_output.ids)
        if hasattr(prompt_output, "ids")
        else list(prompt_output)
    )

    full_ids = (
        list(full_output.ids)
        if hasattr(full_output, "ids")
        else list(full_output)
    )

    # Verify prefix alignment
    assert full_ids[:len(prompt_ids)] == prompt_ids, (
        "Prompt tokens do not match the beginning of full conversation!"
    )

    # Mask prompt tokens
    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

In [20]:
sample_tokens = tokenize_sms_example(
    train_df.iloc[0]["message"],
    train_df.iloc[0]["label"]
)

print("Total tokens:", len(sample_tokens["input_ids"]))
print("Prompt tokens:", sample_tokens["labels"].count(-100))
print(
    "Assistant tokens:",
    len(sample_tokens["labels"]) - sample_tokens["labels"].count(-100)
)

Total tokens: 2
Prompt tokens: 2
Assistant tokens: 0
